# Memoria Técnica: Implementación y Verificación de la CPU Multiciclo S-MIPS

**Autor:** Eduardo Alexis Vazquez Menéndez  
**Grupo:** C211

---

## 1. Autoría y decisión sobre VHDL

La implementación de la CPU multiciclo S-MIPS se ha realizado íntegramente dentro del bloque `CPU` mediante **componentes nativos de Logisim Evolution**, sin emplear código VHDL ni archivos `.vhd` en el proyecto.

**Justificación del enfoque:** trabajar exclusivamente con bloques nativos de Logisim Evolution ha permitido construir el procesador de forma visual, trazar directamente las señales entre el *datapath*, la decodificación y la unidad de control, y comprobar paso a paso el comportamiento de cada bus y registro durante la simulación. Esta aproximación resulta especialmente práctica para depurar transiciones de ciclo y verificar la coherencia de las señales de control sin introducir una capa adicional de traducción a lenguaje hardware.

---

## 2. Arquitectura de la ruta de datos y señales de control

El núcleo del procesador organiza el flujo de datos entre la memoria compartida, los registros de trabajo, la unidad aritmético-lógica (`ALU`) y la lógica de acceso a bytes (`Byte_Handler`). La selección de dirección en memoria depende de `IOR_D`: cuando vale `0`, el bus apunta a `PC` para leer la siguiente instrucción; cuando vale `1`, toma `ALUOut` con la dirección efectiva calculada previamente en `S_MEM_ADDR` (sumando base y desplazamiento).

```text
                       [PC_WRITE, PC_SRC]
                                |
                                v
   +--------+     +----------------------+     +----+     +---------------------+
   |   PC   |---->| Memoria (IOR_D=0/1)  |---->| IR |---->| Instruction_Decoder |
   +--------+     +----------------------+     +----+     +---------------------+
       ^            ^            |               |                  |
       |            |            |           [IR_WRITE]             v
       |            |            |                         I_*, F_*, REG_DST,
       |            |            |                         MEM_TO_REG, EXT_OP
       |            |            |                                  |
       |            |            v                                  v
       |            |        +-------+                    +-------------------+
       |            |        |  MDR  |                    | Banco de registros|
       |            |        +-------+                    +-------------------+
       |            |            |                          |         |
       |            |       [MDR_WRITE]             [A_WRITE]   [B_WRITE]
       |            |            |                          v         v
       |            |            v                        +---+     +---+
       |            |    +---------------+                | A |     | B |
       |            +----| Byte_Handler  |                +---+     +---+
       |                 +---------------+                  |         |
       |                  [MEM_BYTE, I_LB]                  |         |
       |                         |                          +----+----+
       |                         |                               |
       |                         |             [ALU_SRC_A,       |
       |                         |              ALU_SRC_B,       v
       |                         |              ALU_CONTROL]  +-----+
       |                         |                            | ALU |
       |                         |                            +-----+
       |                         |                               |
       |                         |                        [ALU_OUT_WRITE]
       |                         |                               v
       |                         |                            +--------+
       |                         +--------------------------->| ALUOut |
       |                                                    +--------+
       |                                                         |
       +----------------------[PC_SRC = 0]-----------------------+
```

### Catálogo de señales generadas por `Control_Unit`

* **Habilitación de escritura:** `REG_WRITE`, `IR_WRITE`, `PC_WRITE`, `A_WRITE`, `B_WRITE`, `BT_WRITE`, `MDR_WRITE`, `ALU_OUT_WRITE`, `MEM_WE`, `TTY_WE` y `HALT`.
* **Control de rutas y selección:** `PC_SRC` (2 bits), `ALU_SRC_A` (2 bits), `ALU_SRC_B` (2 bits), `IOR_D` (1 bit), `ALU_CONTROL` (3 bits), `EXT_OP` (2 bits), `REG_DST` (2 bits) y `MEM_TO_REG` (2 bits).

---

## 3. Unidad de control: máquina de estados finitos (18 estados)

La secuencia de ejecución se gobierna mediante una máquina de estados finitos de 18 estados. Cada estado dispone de un biestable de 1 bit cuya salida actual se denota como `S_*` y cuya entrada de próximo estado se denota como `N_*`. En cada flanco ascendente de `CLK`, la actualización sigue la regla $S\_X(t+1) \leftarrow N\_X(t)$, con habilitación permanente (`ONE1`) y puesta a cero síncrona/asíncrona mediante `RESET`.

### 3.1 Mapa de estados y condiciones de transición

```text
   (RESET / NO_ACTIVE_STATE)
               |
               v
          [S_FETCH] ----(siempre)----> [S_DECODE]
                                           |
        +------------------+---------------+---------------+-------------------+-------------------+
        |                  |               |               |                   |                   |
   (I_ALU_R)          (I_I_ALU)   (I_LOAD \lor I_STORE) (I_BRANCH)    (I_J, I_JAL, I_JR)    (I_SYSCALL)
        |                  |               |               |                   |                   |
        v                  v               v               v                   |                   v
   [S_EXEC_R]         [S_EXEC_I]      [S_MEM_ADDR]    [S_BRANCH]               |              [S_SYSCALL]
        |                  |            /        \       /      \              |              /    |     \
    (siempre)          (siempre)  (I_LOAD)  (I_STORE)  | (BT=0) \ (BT=1)     |      (V0=10) (V0=11) (otro V0)
        v                  v         /            \    |         v           |         |        |       |
    [S_WB_R]           [S_WB_I]     v              v   |    [S_BRANCH_T]     |         |        v       |
        |                  |    [S_MEM_RD]     [S_MEM_WR]          |           |         |     [S_TTY]    |
        |                  |        |              |   |           |           |         |        |       |
        |                  |    (siempre)          |   |           |           |         |    (siempre)   |
        |                  |        v              |   |           |           |         |        |       |
        |                  |    [S_MEM_WB]         |   |           |           |         |        |       |
        |                  |        |              |   |           |           |         |        |       |
        +------------------+--------+--------------+---+-----------+-----------+---------+--------+-------+
                                                        |                               |
                                                        +-----------------(ILLEGAL)---+
                                                                                        |
                                                                                        v
                                                                              ((  S_HALT  )) <---+
                                                                                   |             |
                                                                                   +---(siempre)---+
```

*(Nota: en el diagrama, `BT` abreviado corresponde a `BRANCH_TAKEN`; las rutas de retorno hacia `S_FETCH` desde `S_WB_R`, `S_WB_I`, `S_MEM_WB`, `S_MEM_WR`, `S_BRANCH_T`, `S_BRANCH` cuando `BRANCH_TAKEN=0`, `S_JUMP`, `S_JAL`, `S_JR`, `S_TTY` y `S_SYSCALL` con otro `V0` convergen en la lógica de retorno a `S_FETCH`).*

### 3.2 Reglas de formación del próximo estado

* **Inicialización tras `RESET`:** al activarse `RESET`, todos los registros `S_*` quedan en `0`. Como `ANY_STATE` vale `0` y `NO_ACTIVE_STATE` vale `1`, se fuerza `N_S_FETCH` para iniciar el ciclo en `S_FETCH`.
* **Condición de retorno a `S_FETCH`:** `N_S_FETCH` resulta de aplicar un OR entre los estados finales de instrucción (`S_WB_R`, `S_WB_I`, `S_MEM_WB`, `S_MEM_WR`, `S_BRANCH_T`, `S_JUMP`, `S_JAL`, `S_JR`, `S_TTY`), la rama no tomada `T_BRANCH_NT` ($= \text{S\_BRANCH} \land \neg\text{BRANCH\_TAKEN}$), la continuación de llamada al sistema `T_SYS_CONT` (cuando `S_SYSCALL` está activo y $V0$ no coincide ni con `10` ni con `11`) y `NO_ACTIVE_STATE`.
* **Tratamiento de `syscall`:** el registro `$v0` se compara contra las constantes `SC_EXIT_10 = 10` y `SC_PUTC_11 = 11`. Si coincide con `11` (`V0_EQ_11`), se activa `N_S_TTY` $= \text{S\_SYSCALL} \land \text{V0\_EQ\_11}$; si coincide con `10` (`V0_EQ_10`), aporta a la entrada de parada.
* **Estado de parada (`S_HALT`):** la señal `N_S_HALT` se construye como $\text{S\_HALT} \lor \text{N\_HALT\_ILL} \lor \text{N\_HALT\_SYS}$, haciendo que una vez alcanzado `S_HALT` la máquina permanezca bloqueada en él de forma absorbente.

---

## 4. Activación de señales de control por estado

En la siguiente tabla se recoge el comportamiento de las señales de control en cada estado. Se utiliza `1` para señal activa, `0` para inactiva, `X` cuando su valor es indiferente en el ciclo actual, y referencias simbólicas o valores de selección (`ADD`, `SUB`, `PC/4`, `A/B`, `A/Imm`, etc.) según el diseño de la unidad de control.

| Estado | `IR_WRITE` | `REG_WRITE` | `PC_WRITE` | `ALU_OUT_WRITE` | `A_WRITE` | `B_WRITE` | `MDR_WRITE` | `MEM_WE` | `TTY_WE` | `IOR_D` | `PC_SRC` | `ALU_SRC_A / B` | `ALU_CONTROL` |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| `S_FETCH` | `1` | `0` | `1` | `0` | `0` | `0` | `0` | `0` | `0` | `0` | `X` | `PC / 4` | `ADD` |
| `S_DECODE` | `0` | `0` | `0` | `0` | `1` | `0` | `0` | `0` | `0` | `0` | `X` | `PC / X` | `ADD` |
| `S_EXEC_R` | `0` | `0` | `0` | `0` | `1` | `0` | `0` | `0` | `0` | `0` | `X` | `A / B` (o `shamt`) | `D` |
| `S_WB_R` | `0` | `1` | `0` | `0` | `0` | `0` | `0` | `0` | `0` | `0` | `X` | `X` | `X` |
| `S_EXEC_I` | `0` | `0` | `0` | `1` | `0` | `0` | `0` | `0` | `0` | `X` | `0` | `A / Imm` | `D` |
| `S_WB_I` | `0` | `1` | `0` | `0` | `0` | `0` | `0` | `0` | `0` | `X` | `X` | `X` | `X` |
| `S_MEM_ADDR` | `0` | `0` | `0` | `0` | `1` | `0` | `0` | `0` | `0` | `X` | `0` | `A / Imm` | `ADD` |
| `S_MEM_RD` | `0` | `0` | `0` | `0` | `0` | `1` | `0` | `0` | `0` | `X` | `1` | `X` | `X` |
| `S_MEM_WB` | `1` | `0` | `0` | `0` | `0` | `0` | `0` | `0` | `0` | `X` | `1` | `X` | `X` |
| `S_MEM_WR` | `0` | `0` | `0` | `0` | `0` | `0` | `0` | `1` | `0` | `1` | `X` | `A / Imm` | `ADD` |
| `S_BRANCH` | `0` | `0` | `0` | `0` | `0` | `0` | `0` | `0` | `0` | `X` | `0` | `A / B` | `SUB` |
| `S_BRANCH_T` | `0` | `0` | `1` | `0` | `0` | `0` | `0` | `0` | `0` | `0` | `1` | `X` | `X` |
| `S_JUMP` | `0` | `0` | `1` | `0` | `0` | `0` | `0` | `0` | `0` | `0` | `2` | `X` | `X` |
| `S_JAL` | `0` | `1` | `1` | `0` | `0` | `0` | `0` | `0` | `0` | `0` | `2` | `X` | `X` |
| `S_JR` | `0` | `0` | `1` | `0` | `0` | `0` | `0` | `0` | `0` | `3` | `0` | `X` | `X` |
| `S_SYSCALL` | `0` | `0` | `0` | `0` | `0` | `0` | `0` | `0` | `0` | `X` | `0` | `X` | `X` |
| `S_TTY` | `0` | `0` | `0` | `0` | `0` | `0` | `0` | `0` | `1` | `0` | `X` | `X` | `X` |
| `S_HALT` | `0` | `0` | `0` | `0` | `0` | `0` | `0` | `0` | `0` | `0` | `X` | `X` | `X` |

### Términos intermedios derivados del estado

Para simplificar la lógica combinacional de selección, la unidad de control define las siguientes señales auxiliares:

* $T\_LS = I\_LOAD \lor I\_STORE$
* $T\_USE\_A = S\_EXEC\_I \lor S\_MEM\_ADDR \lor S\_BRANCH$
* $T\_USE\_PC = S\_FETCH \lor S\_DECODE$
* $T\_USE\_IMM = S\_EXEC\_I \lor S\_MEM\_ADDR$
* $T\_ALU\_ADD = S\_FETCH \lor S\_DECODE \lor S\_MEM\_ADDR$
* $T\_EXT\_SIGN = S\_DECODE \lor S\_MEM\_ADDR \lor S\_BRANCH$
* $SEL\_PC\_JUMP\_JAL = S\_JUMP \lor S\_JAL$

### Encadenamiento de multiplexores y selectores directos

Varias señales de control se obtienen mediante cadenas de multiplexores `2:1` en serie (conectando la salida de una etapa a la entrada `D0` de la siguiente) o mediante selección directa sobre la propuesta del decodificador:

* **`PC_SRC`:** $PC\_01 = S\_BRANCH\_T \text{ ? } 1 : 0 \;\rightarrow\; PC\_012 = SEL\_PC\_JUMP\_JAL \text{ ? } 2 : PC\_01 \;\rightarrow\; PC\_SRC = S\_JR \text{ ? } 3 : PC\_012$
* **`ALU_SRC_A`:** $ASA\_01 = T\_SEL\_A \text{ ? } 1 : 0 \;\rightarrow\; ASA\_012 = T\_SHIFT \text{ ? } 2 : ASA\_01 \;\rightarrow\; ALU\_SRC\_A = T\_USE\_PC \text{ ? } 0 : ASA\_012$
* **`ALU_SRC_B`:** $ASB\_01 = S\_FETCH \text{ ? } 1 : 0 \;\rightarrow\; ASB\_012 = T\_USE\_IMM \text{ ? } 2 : ASB\_01 \;\rightarrow\; ALU\_SRC\_B = S\_DECODE \text{ ? } 3 : ASB\_012$
* **`ALU_CONTROL`:** $AC\_1 = T\_ALU\_ADD \text{ ? } 000 : DEC\_ALU\_CONTROL \;\rightarrow\; ALU\_CONTROL = S\_BRANCH \text{ ? } 001 : AC\_1$
* **Selección directa:**
  * `EXT_OP` $= T\_EXT\_SIGN \text{ ? } 0 : DEC\_EXT\_OP$
  * `REG_DST` $= S\_JAL \text{ ? } 2 : DEC\_REG\_DST$
  * `MEM_TO_REG` $= S\_JAL \text{ ? } 2 : DEC\_MEM\_TO\_REG$

---

## 5. Codificación de la ALU y decodificación de instrucciones

### 5.1 Operaciones de la ALU

La `ALU` emplea un bus de control `ALU_CONTROL` de 3 bits junto con un decodificador `3:8` (`Decoder_3`) y un multiplexor `8:1` (`Multiplexer_3`). Los códigos `000` y `001` comparten la rama del sumador, pero el bit 0 invierte el operando `B` y habilita el acarreo de entrada para ejecutar la resta. Además, un comparador de 32 bits genera la señal `ZERO` cuando el resultado es cero, utilizada por `Branch_Control` junto con `I_BNE` para determinar `BRANCH_TAKEN`.

| `ALU_CONTROL` | Operación realizada | Aplicaciones principales |
| :---: | :--- | :--- |
| `000` | $A + B$ (suma) | Incremento `PC+4`, direcciones efectivas, `add`, `addu`, `addi`, `addiu` |
| `001` | $A - B$ (resta) | `sub` y evaluación de igualdad/desigualdad en bifurcaciones |
| `010` | $A \land B$ | Operaciones lógicas `and` y `andi` |
| `011` | $A \lor B$ | Operaciones lógicas `or` y `ori` |
| `100` | $A < B \text{ ? } 1 : 0$ | Comparaciones con signo `slt` y `slti` |
| `101` | $A \ll \text{shamt}$ | Desplazamiento lógico a la izquierda (`sll`) |
| `110` | $A \gg \text{shamt}$ (lógico) | Desplazamiento lógico a la derecha (`srl`) |
| `111` | $B \ll 16$ | Carga de inmediato superior (`lui`) |

### 5.2 Señales producidas por `Instruction_Decoder`

A partir de los campos `OPCODE` (6 bits) y `FUNCT` (6 bits), el decodificador identifica categorías de instrucción (`I_*`), funciones específicas (`F_*`) y señales auxiliares de control:

| Señal | Categoría | Condición de activación | Función dentro del procesador |
| :--- | :--- | :--- | :--- |
| `I_ALU_R` | Clase | `OPCODE = 0` y `FUNCT` aritmético-lógico | Gobernar ejecución de instrucciones tipo R |
| `I_I_ALU` | Clase | `addi`, `addiu`, `andi`, `ori`, `slti` | Gobernar ejecución de instrucciones aritmético-lógicas tipo I |
| `I_LOAD` | Clase | `lw`, `lb`, `lbu` | Indicar lectura desde memoria |
| `I_STORE` | Clase | `sw`, `sb` | Indicar escritura hacia memoria |
| `I_BRANCH` | Clase | `beq`, `bne` | Evaluar condiciones de salto condicional |
| `I_J`, `I_JAL` | Instrucción | `OPCODE = 2` o `OPCODE = 3` | Seleccionar saltos directos (`j` / `jal`) |
| `I_JR` | Instrucción | `OPCODE = 0` y `FUNCT = 8` | Saltar a la dirección contenida en registro |
| `I_SYSCALL` | Instrucción | `OPCODE = 0` y `FUNCT = 12` | Invocar servicio del sistema |
| `I_SHIFT` | Clase | `sll`, `srl` | Elegir `shamt` como entrada de desplazamiento en la ALU |
| `ILLEGAL` | Estado | Ningún patrón válido reconocido | Provocar transición hacia `S_HALT` |
| `MEM_BYTE` | Dato | `lb`, `lbu`, `sb` | Activar tratamiento a nivel de byte en `Byte_Handler` |
| `F_ADD`, `F_ADDU`, `F_SUB`, `F_AND`, `F_OR`, `F_SLT` | Función (`FUNCT`) | Códigos `32`, `33`, `34`, `36`, `37`, `42` | Identificar la operación R-type concreta |
| `F_SLL`, `F_SRL` | Función (`FUNCT`) | Códigos `0`, `2` | Identificar el tipo de desplazamiento |
| `F_JR`, `F_SYSCALL` | Función (`FUNCT`) | Códigos `8`, `12` | Distinguir `jr` y `syscall` cuando `OPCODE = 0` |

### 5.3 Generación de buses propuestos por el decodificador

El propio `Instruction_Decoder` arma los valores por defecto de `ALU_CONTROL`, `EXT_OP`, `REG_DST` y `MEM_TO_REG` mediante puertas OR y un *splitter* combinador de bits; posteriormente, `Control_Unit` puede sobrescribirlos según el estado activo:

* **`ALU_CONTROL` (3 bits):**
  * Bit 0: $I\_SUB \lor I\_BEQ \lor I\_BNE \lor I\_OR \lor I\_ORI \lor I\_SLL \lor I\_LUI$
  * Bit 1: $I\_AND \lor I\_ANDI \lor I\_OR \lor I\_ORI \lor I\_SRL \lor I\_LUI$
  * Bit 2: $I\_SLT \lor I\_SLTI \lor I\_SLL \lor I\_SRL \lor I\_LUI$
  * Con esta lógica se obtienen, por ejemplo, `000` (`add`), `001` (`sub`), `010` (`and`), `011` (`or`), `100` (`slt`), `101` (`sll`), `110` (`srl`) y `111` (`lui`).
* **`EXT_OP` (2 bits):** bit 0 $= I\_ANDI \lor I\_ORI$ (extensión con ceros para inmediatos lógicos) y bit 1 $= I\_LUI$ (desplazamiento a los 16 bits superiores); con ambos bits en `0` se aplica extensión con signo.
* **`REG_DST` (2 bits):** bit 0 $= I\_ALU\_R$ y bit 1 $= I\_JAL$, determinando si el registro destino proviene de `IR[rd]`, `IR[rt]` o `$ra` (registro 31).
* **`MEM_TO_REG` (2 bits):** bit 0 $= I\_LOAD$ y bit 1 $= I\_JAL$, seleccionando entre el resultado de la ALU, el dato leído en `MDR` o el `PC` de retorno.

---

## 6. Latencia en ciclos por clase de instrucción

Midiendo desde la entrada a `S_FETCH` hasta el regreso a `S_FETCH` (o detención en `S_HALT`), la duración de cada clase de instrucción es la siguiente:

| Clase de instrucción | Secuencia de estados | Ciclos |
| :--- | :--- | :---: |
| Aritmética/lógica tipo R | `S_FETCH -> S_DECODE -> S_EXEC_R -> S_WB_R` | `4` |
| Aritmética/lógica inmediata (tipo I) | `S_FETCH -> S_DECODE -> S_EXEC_I -> S_WB_I` | `4` |
| Lectura en memoria (`lw`, `lb`, `lbu`) | `S_FETCH -> S_DECODE -> S_MEM_ADDR -> S_MEM_RD -> S_MEM_WB` | `5` |
| Escritura en memoria (`sw`, `sb`) | `S_FETCH -> S_DECODE -> S_MEM_ADDR -> S_MEM_WR` | `4` |
| Bifurcación (`beq`, `bne`) | `S_FETCH -> S_DECODE -> S_BRANCH [-> S_BRANCH_T]` | `3` o `4` |
| Salto (`j`, `jal`, `jr`) | `S_FETCH -> S_DECODE -> S_JUMP / S_JAL / S_JR` | `3` |
| `syscall` servicio 11 | `S_FETCH -> S_DECODE -> S_SYSCALL -> S_TTY` | `4` |
| `syscall` servicio 10 | `S_FETCH -> S_DECODE -> S_SYSCALL -> S_HALT` | `4` |

---

## 7. Cobertura de pruebas en `tests/` y programas de validación

La suite de verificación combina los siete programas específicos de prueba (`test01_*` a `test07_*`) con los siete programas de aplicación (`arithmetic.asm`, `array_sum.asm`, `branches.asm`, `gcd.asm`, `hello.asm`, `memory.asm` y `multiply.asm`):

| Instrucción / Requisito | Pruebas en `tests/` | Programas adicionales | Escenarios cubiertos |
| :--- | :--- | :--- | :--- |
| `add`, `sub` | `test01_alu_add_sub.asm` | `arithmetic.asm`, `gcd.asm` | Combinaciones con operandos positivos y negativos (`+1500` y `-500`, `1000` y `-7`, restas con signo positivo y negativo). |
| `addi` | `test01_alu_add_sub.asm` | `arithmetic.asm` | Constantes inmediatas positivas y negativas (`-1005`, `-7`, `100`) con extensión de signo. |
| `addu`, `addiu` | `test01_alu_add_sub.asm` | `arithmetic.asm` | Expansión de `li` (en rango de 16 bits con signo) y `move` sobre registros. |
| `and`, `or` | `test02_logic_lui.asm` | `arithmetic.asm` | Operandos con bits activos en los 16 bits superiores (`0x8000F0F0` y `0xC3C35A5A`). |
| `andi`, `ori` | `test02_logic_lui.asm` | `arithmetic.asm` | Inmediatos con bit 15 igual a `1` (`0xF0F0`, `0x8000`, `0x80FF`), verificando extensión por ceros. |
| `lui` | `test02_logic_lui.asm` | `arithmetic.asm`, `memory.asm` | Inmediatos con bit 15 en `1` (`0x8000`, `0xC3C3`, `0x8001`) ubicados en la mitad superior de palabra. |
| `sll`, `srl` | `test03_shifts_slt.asm` | `arithmetic.asm`, `array_sum.asm` | Desplazamientos nulos (`shamt = 0`) y no nulos (`shamt != 0`), comprobando relleno con ceros en `srl` sobre valores negativos. |
| `slt`, `slti` | `test03_shifts_slt.asm` | `arithmetic.asm`, `branches.asm` | Salidas `0` y `1` en comparaciones con signo frente a valores negativos (`-16 < 5`, `-16 < 10`, `5 < -1`). |
| `lw`, `sw` | `test04_mem_words.asm` | `memory.asm`, `array_sum.asm` | Accesos a palabra con desplazamientos negativos y positivos (`-8`, `-4`, `4`, `12`) y lectura posterior de comprobación. |
| `lb`, `lbu` | `test05_mem_bytes.asm` | `memory.asm`, `hello.asm` | Lectura de bytes con bit 7 en `1` (`0x81`, `0xFE`, `0xA5`, `0x80`, `0xe9`) en distintas posiciones (`d[1:0]`). |
| `sb` | `test05_mem_bytes.asm` | `memory.asm` | Escritura en los cuatro bytes de una palabra (`d[1:0] = 0, 1, 2, 3`), verificando con `lw` que no se alteran los demás bytes. |
| `beq`, `bne` | `test06_branches_jumps.asm` | `branches.asm`, `gcd.asm` | Saltos condicionales tomados y no tomados, saltos hacia adelante y saltos hacia atrás en bucles. |
| `j` | `test06_branches_jumps.asm` | `branches.asm`, `gcd.asm` | Saltos incondicionales directos y control de flujo en bucles. |
| `jal`, `jr` | `test07_calls_zero.asm` | `branches.asm`, `gcd.asm`, `multiply.asm` | Llamadas a procedimientos no hoja con salvado y restauración de `$ra` y argumentos en pila (`factorial` recursivo). |
| `syscall` (11 y 10) | `test01_alu_add_sub.asm` – `test07_calls_zero.asm` | `hello.asm`, `array_sum.asm`, `memory.asm` | Salida de caracteres/dígitos (`11`) y finalización limpia de ejecución (`10`). |
| `$zero` | `test07_calls_zero.asm` | `arithmetic.asm` | Intentos de escritura sobre `$zero` (`addi`, `add`, `ori`) y comprobación posterior de su valor constante `0`. |

---

## 8. Resultados de `run_tests.py` y observaciones finales

Al ejecutar `python tools/run_tests.py` sobre el diseño completo, los 14 programas de prueba superaron la validación sin discrepancias:

```text
Ejecutando 14 programas en la placa...
OK    programs/arithmetic.asm  (6664 ciclos)
OK    programs/array_sum.asm  (915 ciclos)
OK    programs/branches.asm  (926 ciclos)
OK    programs/gcd.asm  (2851 ciclos)
OK    programs/hello.asm  (297 ciclos)
OK    programs/memory.asm  (5075 ciclos)
OK    programs/multiply.asm  (1971 ciclos)
OK    programs/test01_alu_add_sub.asm  (2178 ciclos)
OK    programs/test02_logic_lui.asm  (1801 ciclos)
OK    programs/test03_shifts_slt.asm  (1530 ciclos)
OK    programs/test04_mem_words.asm  (1480 ciclos)
OK    programs/test05_mem_bytes.asm  (2890 ciclos)
OK    programs/test06_branches_jumps.asm  (461 ciclos)
OK    programs/test07_calls_zero.asm  (1179 ciclos)

14/14 correctos
```

### Limitaciones y errores conocidos

* **Limitaciones del entorno y la arquitectura:** la placa soporta únicamente los servicios `10` (fin de ejecución) y `11` (impresión de carácter) de `syscall`, por lo que las rutinas de impresión de cadenas, enteros en decimal o valores hexadecimales se implementan en software dentro de los propios programas. Asimismo, el procesador se ha construido.
* **Errores conocidos:** no se han observado fallos ni discrepancias en los 14 programas de prueba ejecutados (`14/14` correctos). 